In [ ]:
```xml
<VSCode.Cell language="markdown">
# 📡 Event-Driven Architecture 2.0 Guide

**Advanced event sourcing, event store patterns, and temporal event processing**

This guide covers:
- Event sourcing principles
- Event store architecture
- Event versioning
- Event upcasting
- Temporal queries
- Snapshots
- Event projections
- Saga pattern with events
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 1. Event Store & Sourcing

### Event Store Implementation
```python
from dataclasses import dataclass, field
from typing import Dict, Optional, List, Any
from datetime import datetime
from enum import Enum
import json

class EventType(Enum):
    CREATED = "created"
    UPDATED = "updated"
    DELETED = "deleted"
    ACTIVATED = "activated"
    DEACTIVATED = "deactivated"

@dataclass
class Event:
    """Domain event"""
    event_id: str
    event_type: str
    aggregate_id: str
    aggregate_type: str
    version: int
    timestamp: datetime
    data: Dict
    metadata: Dict = field(default_factory=dict)
    correlationId: Optional[str] = None

@dataclass
class EventMetadata:
    """Event metadata"""
    causation_id: str
    correlation_id: str
    actor: str
    source: str
    timestamp: datetime

class EventStore:
    """Event sourced event store"""
    
    def __init__(self):
        self.events: List[Event] = []
        self.snapshots: Dict[str, Dict] = {}
        self.event_index: Dict[str, List[Event]] = {}
    
    def append_event(self, event: Event) -> bool:
        """Append event to store"""
        
        # Assign sequence number
        event.version = len(self.events) + 1
        event.timestamp = datetime.utcnow()
        
        self.events.append(event)
        
        # Index by aggregate ID
        if event.aggregate_id not in self.event_index:
            self.event_index[event.aggregate_id] = []
        
        self.event_index[event.aggregate_id].append(event)
        
        return True
    
    def get_events(self, aggregate_id: str, from_version: int = 0) -> List[Event]:
        """Get events for aggregate"""
        
        if aggregate_id not in self.event_index:
            return []
        
        return [e for e in self.event_index[aggregate_id] if e.version > from_version]
    
    def rebuild_aggregate(self, aggregate_id: str) -> Dict:
        """Rebuild aggregate from events"""
        
        # Check for snapshot first
        if aggregate_id in self.snapshots:
            snapshot = self.snapshots[aggregate_id]
            from_version = snapshot['version']
            state = snapshot['state']
        else:
            from_version = 0
            state = {'id': aggregate_id, 'version': 0}
        
        # Apply events after snapshot
        events = self.get_events(aggregate_id, from_version)
        
        for event in events:
            state = self._apply_event(state, event)
        
        return state
    
    def _apply_event(self, state: Dict, event: Event) -> Dict:
        """Apply event to state"""
        
        if event.event_type == 'created':
            return {**state, **event.data}
        elif event.event_type == 'updated':
            return {**state, **event.data}
        elif event.event_type == 'deleted':
            return {**state, 'deleted': True}
        
        return state
    
    def create_snapshot(self, aggregate_id: str, version: int, state: Dict) -> bool:
        """Create snapshot"""
        
        self.snapshots[aggregate_id] = {
            'version': version,
            'state': state,
            'created_at': datetime.utcnow().isoformat()
        }
        
        return True
    
    def get_all_events(self) -> List[Event]:
        """Get all events"""
        return sorted(self.events, key=lambda e: e.version)

class EventProjection:
    """Event projection to read model"""
    
    def __init__(self):
        self.projections: Dict[str, Dict] = {}
    
    def project_event(self, event: Event):
        """Project event to read model"""
        
        if event.aggregate_id not in self.projections:
            self.projections[event.aggregate_id] = {}
        
        projection = self.projections[event.aggregate_id]
        
        # Update projection based on event
        if event.event_type == 'created':
            projection.update(event.data)
        elif event.event_type == 'updated':
            projection.update(event.data)
        elif event.event_type == 'deleted':
            projection['deleted'] = True
        
        projection['last_updated'] = event.timestamp.isoformat()
        projection['version'] = event.version
    
    def get_projection(self, aggregate_id: str) -> Optional[Dict]:
        """Get projection"""
        
        return self.projections.get(aggregate_id)
    
    def query_projections(self, filters: Dict = None) -> List[Dict]:
        """Query projections"""
        
        results = list(self.projections.values())
        
        if not filters:
            return results
        
        # Apply filters
        for key, value in filters.items():
            results = [r for r in results if r.get(key) == value]
        
        return results
```
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 2. Event Versioning & Upcasting

### Event Version Management
```python
class EventVersion:
    """Event version"""
    
    def __init__(self, event_type: str, version: int, schema: Dict):
        self.event_type = event_type
        self.version = version
        self.schema = schema

class EventUpcaster:
    """Upcast old events to new versions"""
    
    def __init__(self):
        self.upcasters: Dict[tuple, callable] = {}
        self.event_versions: Dict[str, List[EventVersion]] = {}
    
    def register_upcast(self, event_type: str, from_version: int, 
                       to_version: int, upcast_func: callable):
        """Register upcast function"""
        
        key = (event_type, from_version, to_version)
        self.upcasters[key] = upcast_func
    
    def upcast_event(self, event: Event, target_version: int) -> Event:
        """Upcast event to target version"""
        
        current_version = event.version
        
        while current_version < target_version:
            next_version = current_version + 1
            key = (event.event_type, current_version, next_version)
            
            if key not in self.upcasters:
                break
            
            upcast_func = self.upcasters[key]
            event.data = upcast_func(event.data)
            current_version = next_version
        
        return event
    
    def register_version(self, event_version: EventVersion):
        """Register event version"""
        
        event_type = event_version.event_type
        
        if event_type not in self.event_versions:
            self.event_versions[event_type] = []
        
        self.event_versions[event_type].append(event_version)

# Example upcasters
def upcast_user_created_v1_to_v2(data: Dict) -> Dict:
    """Upcast UserCreated from v1 to v2"""
    # v1: {name, email}
    # v2: {firstName, lastName, email}
    
    name = data.get('name', '')
    parts = name.split(' ', 1)
    
    return {
        'firstName': parts[0],
        'lastName': parts[1] if len(parts) > 1 else '',
        'email': data.get('email', '')
    }

def upcast_user_created_v2_to_v3(data: Dict) -> Dict:
    """Upcast UserCreated from v2 to v3"""
    # v3: adds status field
    
    return {
        **data,
        'status': 'active'
    }
```

### Temporal Queries
```python
class TemporalQuery:
    """Query events at point in time"""
    
    def __init__(self, event_store: EventStore):
        self.event_store = event_store
    
    def get_aggregate_at_time(self, aggregate_id: str, timestamp: datetime) -> Dict:
        """Get aggregate state at specific time"""
        
        all_events = self.event_store.get_events(aggregate_id)
        
        # Filter events before timestamp
        events_at_time = [e for e in all_events if e.timestamp <= timestamp]
        
        # Rebuild state from filtered events
        state = {'id': aggregate_id}
        
        for event in events_at_time:
            state = self.event_store._apply_event(state, event)
        
        return state
    
    def get_aggregate_between(self, aggregate_id: str, 
                             start_time: datetime, end_time: datetime) -> List[Dict]:
        """Get aggregate changes between times"""
        
        all_events = self.event_store.get_events(aggregate_id)
        
        # Filter events in time range
        events_in_range = [e for e in all_events 
                          if start_time <= e.timestamp <= end_time]
        
        snapshots = []
        state = {'id': aggregate_id}
        
        for event in events_in_range:
            state = self.event_store._apply_event(state, event)
            snapshots.append({
                'timestamp': event.timestamp.isoformat(),
                'state': state.copy()
            })
        
        return snapshots
```
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 3. Saga Pattern

### Saga Orchestrator
```python
class SagaStep:
    """Saga step"""
    
    def __init__(self, name: str, action: callable, compensation: callable = None):
        self.name = name
        self.action = action
        self.compensation = compensation or (lambda x: True)

class SagaOrchestrator:
    """Coordinate distributed transactions"""
    
    def __init__(self):
        self.sagas: Dict[str, List[SagaStep]] = {}
        self.saga_state: Dict[str, Dict] = {}
    
    def define_saga(self, saga_name: str, steps: List[SagaStep]):
        """Define saga"""
        self.sagas[saga_name] = steps
    
    def execute_saga(self, saga_name: str, data: Dict) -> bool:
        """Execute saga with compensation on failure"""
        
        if saga_name not in self.sagas:
            return False
        
        steps = self.sagas[saga_name]
        saga_id = saga_name
        
        self.saga_state[saga_id] = {
            'executed_steps': [],
            'failed': False,
            'data': data
        }
        
        for step in steps:
            try:
                # Execute step
                result = step.action(data)
                
                if not result:
                    raise Exception(f"Step {step.name} failed")
                
                self.saga_state[saga_id]['executed_steps'].append(step.name)
            
            except Exception as e:
                print(f"Saga step failed: {e}")
                self.saga_state[saga_id]['failed'] = True
                
                # Compensate
                for executed_step_name in reversed(self.saga_state[saga_id]['executed_steps']):
                    for step in steps:
                        if step.name == executed_step_name:
                            step.compensation(data)
                            break
                
                return False
        
        return True
```
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 4. Event-Driven Architecture Checklist

✅ **Event Store**
- [ ] Event sourcing implemented
- [ ] Events persisted
- [ ] Aggregate rebuilding
- [ ] Snapshots created
- [ ] Indexes configured

✅ **Event Versioning**
- [ ] Versions tracked
- [ ] Upcasters defined
- [ ] Backward compatibility
- [ ] Migration tested
- [ ] Schema documented

✅ **Projections**
- [ ] Read models updated
- [ ] Query performance
- [ ] Consistency handled
- [ ] Rebuilding strategy
- [ ] Monitoring alerts

✅ **Temporal Queries**
- [ ] Time-travel queries
- [ ] Historical snapshots
- [ ] Audit trails
- [ ] Compliance tracking
- [ ] Performance optimized

✅ **Operational**
- [ ] Event streaming
- [ ] Dead letter handling
- [ ] Replay capability
- [ ] Disaster recovery
- [ ] Documentation
</VSCode.Cell>
```